In [25]:
print('==> Preparing data.............................')
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import datasets, transforms
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
import os
import torch
import numpy as np
from PIL import Image
from torchvision import transforms
from torchvision.datasets import ImageFolder
from torchvision.datasets import MNIST, CIFAR10, CIFAR100, SVHN

import os
import torch
from torch import nn,optim
import torch.nn.functional as F

from torchvision import datasets, transforms

from time import perf_counter

import  numpy as np
import torch.utils.data as Data

from torch.utils.data import Dataset, DataLoader

class Safeman(Dataset):
    
    def __init__(self, data,targets):
        super(Safeman, self).__init__()
        self.data = data
        self.targets = targets
        
     
    def __len__(self):
        return len(self.targets)

    def __getitem__(self, idx):
        img, target = self.data[idx], self.targets[idx]
        return img, target


        
        
        
class Safeman_Filter(Safeman):   #bk by zhongying 0717

    def __Filter__(self, known):
        targets = self.targets.data.numpy()
        mask, new_targets = [], []
        for i in range(len(targets)):
            if targets[i] in known:
                mask.append(i)
                new_targets.append(known.index(targets[i]))
        self.targets = np.array(new_targets)
        mask = torch.tensor(mask).long()
        self.data = torch.index_select(self.data, 0, mask)
        
        
class Safeman_FilterF(Safeman):

    def __Filter__(self, known):
        targets = self.targets.data.numpy()
        mask, new_targets = [], []
        for i in range(len(targets)):
            if targets[i] in known:
                mask.append(i)
                dd = known.index(targets[i])
                if dd == 0:
                    new_targets.append(0)
                else:
                    new_targets.append(1)                   
                    
                #new_targets.append(known.index(targets[i]))
        self.targets = np.array(new_targets)
        mask = torch.tensor(mask).long()
        self.data = torch.index_select(self.data, 0, mask)     
        
        
        
        
class Safeman_FilterB(Safeman):

    def __Filter__(self, known):
        targets = self.targets.data.numpy()
        new_targets = []
        for i in range(len(targets)):
            if targets[i] in known:
                new_targets.append(0)
            else:
                new_targets.append(1)
        self.targets = np.array(new_targets)
        self.data = self.data

class Safeman_FilterC(Safeman):
    
    def __Filter__(self, trainknown):
        train_class_num=len(trainknown)
        for i in range(0,len(self.targets)) :
            if self.targets[i]>train_class_num:
                self.targets[i] = train_class_num
        self.data = self.data



        
def setup_seed(seed):

    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    torch.backends.cudnn.deterministic = True

setup_seed(8)



known=[0, 1, 2,3,4,5,6,7,8,9]
num_class=len(known)


x_train = np.load('./data/sCX_train_kitsune3n28.npy')
x_test = np.load('./data/sCX_final_test_kitsune3n28.npy')
y_train = np.load('./data/sCy_train_kitsune3n28.npy')
y_test = np.load('./data/sCy__final_test_kitsune3n28.npy')

print(x_train.shape, x_test.shape, y_train.shape,y_test.shape)


train_dataset = Data.TensorDataset(torch.tensor(x_train), torch.tensor(y_train))
train_dataset.data = train_dataset.tensors[0]
train_dataset.targets = train_dataset.tensors[1]


test_dataset = Data.TensorDataset(torch.tensor(x_test), torch.tensor(y_test))
test_dataset.data = test_dataset.tensors[0]
test_dataset.targets = test_dataset.tensors[1]

labels=['0','1']


train_dataset.classes = labels
test_dataset.classes = labels

train_dataset.classes_to_idx = {i: label for i, label in enumerate(labels)}
test_dataset.classes_to_idx = {i: label for i, label in enumerate(labels)}


b_s=128

num_class=len(labels)

train_loaderxuanze = torch.utils.data.DataLoader(train_dataset, batch_size=b_s, shuffle=True,
    num_workers=4,drop_last=True)

test_loaderxuanze = torch.utils.data.DataLoader(test_dataset, batch_size=b_s, shuffle=True,
    num_workers=4,drop_last=True)

print("done!")



==> Preparing data.............................
(9000, 1, 28, 28) (1000, 1, 28, 28) (9000,) (1000,)
done!


In [26]:
testsetA_data=test_dataset.data
testsetA_targets=test_dataset.targets

In [27]:
x_train_tempte=[]
y_train_tempte=[]
for i in range(len(testsetA_targets)):
    y_train_tempte+=[int(testsetA_targets[i])]
    a = np.resize(testsetA_data[i], (100))
    x_train_tempte += [a]

In [28]:
x_test21, y_test21 = torch.Tensor(x_train_tempte), torch.Tensor(y_train_tempte)

print(x_test21.shape, y_test21.shape)

test_dataset2 = Data.TensorDataset(x_test21, y_test21)
test_dataset2.data = test_dataset2.tensors[0]
test_dataset2.targets = test_dataset2.tensors[1]


test_loader2 = torch.utils.data.DataLoader(
    test_dataset2, batch_size=b_s, shuffle=True,
    num_workers=4,drop_last=True)


print('Real train Data2:', len(test_dataset2))

torch.Size([1000, 100]) torch.Size([1000])
Real train Data2: 1000


In [29]:
trainset=train_dataset

unique,counts = np.unique(trainset.targets,return_counts=True)
print(unique, counts)

[0 1] [7306 1694]


In [30]:
X_trainset_data=trainset.data
X_trainset_targets=trainset.targets

In [31]:
unique,counts = np.unique(X_trainset_targets,return_counts=True)
print(unique, counts)

[0 1] [7306 1694]


In [32]:
X_trainset_targets

tensor([1, 1, 1,  ..., 0, 0, 0])

In [33]:
y_train2=[]

count=[0 ,6]
num_class=2
lists = [[] for i in range(num_class)]
y_train_temp=[]
x_train_temp=[]

In [34]:
X_trainset_data.shape

torch.Size([9000, 1, 28, 28])

In [35]:
X_trainset_targets.shape

torch.Size([9000])

In [36]:
for i in range(len(X_trainset_targets)):
    if len(lists[int(X_trainset_targets[i])])<count[int(X_trainset_targets[i])]:
        lists[int(X_trainset_targets[i])].append(int(X_trainset_targets[i]))   
        a = np.resize(X_trainset_data[i], (3, 32, 32))        
        x_train_temp += [a]
        y_train_temp+=[int(X_trainset_targets[i])]


In [37]:
int(X_trainset_targets[i])

0

In [38]:
len(x_train_temp)

6

In [39]:
unique,counts = np.unique(y_train_temp,return_counts=True)
print(unique, counts)

[1] [6]


In [40]:
len(x_train_temp)

6

In [41]:
trainset.data.shape

torch.Size([9000, 1, 28, 28])

In [42]:
x_train2, y_train2 = torch.Tensor(x_train_temp), torch.Tensor(y_train_temp)

print(x_train2.shape, y_train2.shape)

train_dataset2 = Data.TensorDataset(x_train2, y_train2)
train_dataset2.data = train_dataset2.tensors[0]
train_dataset2.targets = train_dataset2.tensors[1]



b_s=1

train_loader2 = torch.utils.data.DataLoader(
    train_dataset2, batch_size=b_s, shuffle=True,
    num_workers=4,drop_last=True)


torch.Size([6, 3, 32, 32]) torch.Size([6])


In [43]:
from __future__ import print_function, division
import os, random, time, copy
import numpy as np
import pandas as pd
import os.path as path
import scipy.io as sio
from scipy import misc
from scipy import ndimage, signal
import scipy
import pickle
import sys
import math
import matplotlib.pyplot as plt
import PIL.Image
from io import BytesIO


import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.optim as optim
from torch.optim import lr_scheduler 
import torch.nn.functional as F
from torch.autograd import Variable
import torchvision
from torchvision import datasets, models, transforms
import torchvision.utils as vutils


import warnings 
warnings.filterwarnings("ignore")
print(sys.version)
print(torch.__version__)


manualSeed = 999
print("Random Seed: ", manualSeed)
random.seed(manualSeed)
torch.manual_seed(manualSeed)

3.10.19 (main, Oct 21 2025, 16:43:05) [GCC 11.2.0]
2.5.1+cu121
Random Seed:  999


In [44]:
torch.manual_seed(0)

exp_dir = './res1030' 

modelFlag = 'Res18sc'

project_name = 'ukm1030_GANfea_v1_' + modelFlag

device ='cpu'
if torch.cuda.is_available(): 
    device='cuda:0'


total_epoch_num = 5
batch_size = 16    
insertConv = False    
embDimension = 64
isPretrained = False

newsize = (64, 64)


bestEpoch = 2  



path_to_feats = './feats' # the path to cached off-the-shelf features
pklName = path.join(path_to_feats, modelFlag.lower()+'.pkl')

nc = 3
nz = 100
ngf = 64
ndf = 64
beta1 = 0.5
ngpu = 1




nClassTotal = 200
nClassCloseset = nClassTotal

lr = 0.0001 

num_epochs = total_epoch_num
torch.cuda.device_count()
torch.cuda.empty_cache()

save_dir="./res1030/dcgan2nslmal"

print(save_dir)    
if not os.path.exists(save_dir): os.makedirs(save_dir)

log_filename = os.path.join(save_dir, 'train.log')

./res1030/dcgan2nslmal


In [45]:
      
class Generatorzy(nn.Module):
    def __init__(self, z_dim):
        super(Generatorzy, self).__init__()
        
        self.fc = nn.Linear(z_dim, 256*8*8)
        self.g_deconv_1 = nn.Sequential(
                          nn.ConvTranspose2d(256, 128, kernel_size=3,
                                    stride= 2, padding=(3-2+1)//2,
                                    output_padding = (3-2)%2), 
                          nn.BatchNorm2d(128),
                          nn.LeakyReLU()
                          )
        self.g_deconv_2 = nn.Sequential(
                          nn.ConvTranspose2d(128, 64, kernel_size=3,
                                    stride= 1, padding=(3-1+1)//2,
                                    output_padding = (3-1)%2), 
                          nn.BatchNorm2d(64),
                          nn.LeakyReLU()
                          )
        self.g_deconv_3 = nn.Sequential(
                          nn.ConvTranspose2d(64, 3, kernel_size=3,
                                    stride= 2, padding=(3-2+1)//2,
                                    output_padding = (3-2)%2),
                          nn.Tanh()
                          )
        self.fczy = nn.Linear(3*32*32, 100)  #zhongying 0814
        

    def forward(self, x):
        x = self.fc(x).view(-1, 256, 8, 8)
        x = self.g_deconv_1(x)
        x = self.g_deconv_2(x)
        x = self.g_deconv_3(x)
        x_zy = self.fczy(x.view(-1,3*32*32))
        return x_zy,x     


    
class Discriminatorzy(nn.Module):
    def __init__(self):
        super(Discriminatorzy, self).__init__()
        
        self.d_conv_1 = nn.Sequential(
                          nn.Conv2d(3, 32, kernel_size=3,
                                    stride=2, padding=1), 
                          nn.LeakyReLU()
                          )
        self.d_conv_2 = nn.Sequential(
                          nn.Conv2d(32, 64, kernel_size=3,
                                    stride=2, padding=1), 
                          nn.LeakyReLU()
                          )
        self.d_conv_3 = nn.Sequential(
                          nn.Conv2d(64, 128, kernel_size=3,
                                    stride=2, padding=0), 
                          nn.LeakyReLU()
                          )
        self.fc = nn.Linear(3*3*128, 1)
        self.fczy = nn.Linear(3*3*128, 46)  #zhongying 0814

    def forward(self, x):
        x = self.d_conv_1(x)
        x = self.d_conv_2(x)
        x = self.d_conv_3(x)
        x = x.view(-1, 128*3*3)
        x_zy = self.fczy(x)
        x = torch.sigmoid(self.fc(x))
        return x_zy,x

In [46]:
print(device)

def weights_init(m):
    classname = m.__class__.__name__
    if classname.find('Conv') != -1:
        nn.init.normal_(m.weight.data, 0.0, 0.02)
    elif classname.find('BatchNorm') != -1:
        nn.init.normal_(m.weight.data, 1.0, 0.02)
        nn.init.constant_(m.bias.data, 0)     
    

z_dim=100
batch_size=b_s
discriminator = Discriminatorzy()
netDzy = discriminator.to(device)

discriminatoraux = Discriminatorzy()
netDzyaux = discriminatoraux.to(device)

generator = Generatorzy(z_dim)
netGzy = generator.to(device)


netDzy.apply(weights_init)

netDzyaux.apply(weights_init)

netGzy.apply(weights_init)


noise = torch.randn(batch_size, z_dim, device=device)#64--->1 64---->1

_,fake = netGzy(noise)
_,predLabel = netDzy(fake)

print(noise.shape, fake.shape, predLabel.shape)

cuda:0
torch.Size([1, 100]) torch.Size([1, 3, 32, 32]) torch.Size([1, 1])


In [47]:
criterion = nn.BCELoss()


fixed_noise = torch.randn(batch_size, z_dim, device=device)

real_label = 1
fake_label = 0

optimizerD = optim.Adam(netDzy.parameters(), lr=lr/1.5, betas=(beta1, 0.999))

optimizerDaux = optim.Adam(netDzyaux.parameters(), lr=lr/1.5, betas=(beta1, 0.999))

optimizerG = optim.Adam(netGzy.parameters(), lr=lr, betas=(beta1, 0.999))

In [48]:
iters = 0
dataloader_train_closeset=train_loader2
print("Starting Training Loop...")
for epoch in range(num_epochs):
    i=0
    for sample in dataloader_train_closeset:
        data, datalabel = sample
        ############################
        # (1) Update D network: maximize log(D(x)) + log(1 - D(G(z)))
        ###########################
        netDzy.zero_grad()

        # Format batch
        real_cpu = data.to(device)
        bb_size = real_cpu.size(0)
        label = torch.full((bb_size,), real_label, device=device)

        _,output = netDzy(real_cpu)
        output=output.view(-1)

        output=output.to(torch.float32)    
    
        label=label.to(torch.float32)

        
        errD_real = criterion(output, label)

        errD_real.backward()
        

        
        D_x = output.mean().item()

        noise = torch.randn(batch_size, z_dim, device=device)
        _,fake = netGzy(noise)
        label.fill_(fake_label)
        

        _,output = netDzy(fake.detach())
        
        output=output.view(-1)
        output=output.to(torch.float32)
        
        
        label=label.to(torch.float32)

        
        errD_fake = criterion(output, label)

        errD_fake.backward()
        

        
        D_G_z1 = output.mean().item()
        errD = errD_real + errD_fake
        optimizerD.step()

        

        ############################
        # (2) Update G network: maximize log(D(G(z)))
        ###########################
        netGzy.zero_grad()
        label.fill_(real_label)  
        

        _,output = netDzy(fake)
        output=output.view(-1)
        output=output.to(torch.float32)
        label=label.to(torch.float32)
        errG = criterion(output, label)
        

        errGqiuhe=errG
        errGqiuhe.backward()
        
        
        D_G_z2 = output.mean().item()
        optimizerG.step()

        if i % 20 == 0:
            print('[%d/%d][%d/%d]\tLoss_D: %.4f\tLoss_G: %.4f\tD(x): %.4f\tD(G(z)): %.4f / %.4f'
                  % (epoch, num_epochs, i, len(dataloader_train_closeset),
                     errD.item(), errG.item(), D_x, D_G_z1, D_G_z2))

        if (iters % 50 == 0) or ((epoch == num_epochs-1) and (i == len(dataloader_train_closeset)-1)):
            with torch.no_grad():
                pass

        iters += 1
        i+=1
        
        
    cur_model_wts = copy.deepcopy(netGzy.state_dict())
    path_to_save_paramOnly = os.path.join(save_dir, 'dcgan-2nsl-epoch-{}.GNet'.format(epoch+1))
    torch.save(cur_model_wts, path_to_save_paramOnly)
    
    cur_model_wts = copy.deepcopy(netDzy.state_dict())
    path_to_save_paramOnly = os.path.join(save_dir, 'dcgan-2nsl-epoch-{}.DNet'.format(epoch+1))
    torch.save(cur_model_wts, path_to_save_paramOnly)


Starting Training Loop...
[0/5][0/6]	Loss_D: 1.3878	Loss_G: 0.6902	D(x): 0.5008	D(G(z)): 0.5016 / 0.5015
[1/5][0/6]	Loss_D: 1.3843	Loss_G: 0.6908	D(x): 0.5023	D(G(z)): 0.5013 / 0.5012
[2/5][0/6]	Loss_D: 1.3800	Loss_G: 0.6927	D(x): 0.5036	D(G(z)): 0.5004 / 0.5002
[3/5][0/6]	Loss_D: 1.3763	Loss_G: 0.6937	D(x): 0.5050	D(G(z)): 0.5000 / 0.4997
[4/5][0/6]	Loss_D: 1.3717	Loss_G: 0.6943	D(x): 0.5070	D(G(z)): 0.4996 / 0.4994
